# 04 — Entrenar MLP ① (zona de cancha del jugador)

Clasifica cada bbox de jugador en 3 zonas: **lejos / medio / cerca** respecto a la cámara.
Las etiquetas se generan automáticamente del cy del bbox — sin etiquetado manual.

Output: `weights/action_mlp.pt`. Entrenamiento ~2-3 minutos en T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf volley-ai
!git clone https://github.com/Angelote567/DeepLearning.git volley-ai
%cd volley-ai
!pip install -q torch torchvision Pillow tqdm
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/volleyball-2'):
    os.symlink('/content/drive/MyDrive/volley-ai/data/volleyball-2', 'data/volleyball-2')
if os.path.exists('weights') and not os.path.islink('weights'):
    if not os.listdir('weights') or os.listdir('weights') == ['.gitkeep']:
        import shutil; shutil.rmtree('weights')
if not os.path.exists('weights'):
    os.symlink('/content/drive/MyDrive/volley-ai/weights', 'weights')

In [ ]:
from src.train.train_mlp import train
best = train(epochs=40, batch_size=64, lr=1e-3)
print('Mejores pesos:', best)

In [ ]:
# Evaluación rápida — matriz de confusión sobre el split valid
import torch
from torch.utils.data import DataLoader
from src.data.action_dataset import ZoneDataset
from src.models.mlp import ActionMLP
from src.config import ZONE_CLASSES

model = ActionMLP().cuda().eval()
ckpt = torch.load('weights/action_mlp.pt', map_location='cuda')
model.load_state_dict(ckpt['model'])

val_ds = ZoneDataset('data/volleyball-2', split='valid')
val_dl = DataLoader(val_ds, batch_size=64)

n_classes = len(ZONE_CLASSES)
cm = torch.zeros((n_classes, n_classes), dtype=torch.long)
with torch.no_grad():
    for feats, labels in val_dl:
        preds = model(feats.cuda()).argmax(-1).cpu()
        for t, p in zip(labels, preds):
            cm[t.item(), p.item()] += 1

print('Matriz de confusión (rows=true, cols=pred):')
print('         ', '  '.join(f'{c:>6}' for c in ZONE_CLASSES))
for i, c in enumerate(ZONE_CLASSES):
    print(f'{c:>10}', '  '.join(f'{n:>6}' for n in cm[i].tolist()))
print(f'\nAccuracy: {cm.diag().sum().item() / cm.sum().item():.3f}')